# Comparing rich approaches for the same dashboard

> L3 notebook: Table vs Panel+Layout vs static snapshots for the same live-updating dashboard.

This notebook compares three ways to render the same service-status dashboard using Rich. The data is identical across approaches; the trade-offs are in layout control, refresh cost, and export friendliness.

## The shared dataset

All three cells below consume the same list of dicts. Keeping the data in one place makes the comparison fair.

In [ ]:
from rich.console import Console
from rich.panel import Panel
from rich.layout import Layout
from rich.table import Table
from rich.live import Live
from rich.text import Text
from datetime import datetime
import time

console = Console()

checks = [
    {"service": "auth",    "latency_ms": 42,  "status": "ok"},
    {"service": "billing", "latency_ms": 128, "status": "warn"},
    {"service": "search",  "latency_ms": 15,  "status": "ok"},
    {"service": "ingest",  "latency_ms": 310, "status": "err"},
]

## Approach 1 — Table only

A single `Table` is the fastest path. It prints a grid with headers and rows, and it plays nicely with pagers and pipes. The limitation is that the table is a flat block; you cannot overlay panels or split the terminal into regions without leaving the table context.

In [ ]:
def render_table(data, refresh_count=0):
    tbl = Table(title=f"service health (refresh {refresh_count})")
    tbl.add_column("service", style="cyan")
    tbl.add_column("latency_ms", justify="right", style="magenta")
    tbl.add_column("status", justify="center")

    for row in data:
        color = {"ok": "green", "warn": "yellow", "err": "red"}[row["status"]]
        tbl.add_row(
            row["service"],
            str(row["latency_ms"]),
            Text(row["status"], style=color),
        )
    return tbl

console.clear()
console.print(render_table(checks, refresh_count=0))

## Approach 2 — Panel + Layout

`Panel` and `Layout` let you compose multiple renderables into a dashboard with headers, borders, and split regions. Wrapping the same table in a panel and adding a timestamp footer produces a layout that reads more like a status board, but the code path is longer and the live-refresh loop needs `Live` to redraw the whole layout.

In [ ]:
def render_panel_layout(data, refresh_count=0):
    tbl = Table(show_header=True, header_style="bold")
    tbl.add_column("service", style="cyan")
    tbl.add_column("latency_ms", justify="right", style="magenta")
    tbl.add_column("status", justify="center")

    for row in data:
        color = {"ok": "green", "warn": "yellow", "err": "red"}[row["status"]]
        tbl.add_row(
            row["service"],
            str(row["latency_ms"]),
            Text(row["status"], style=color),
        )

    layout = Layout()
    layout.split(
        Layout(name="header", size=3),
        Layout(name="body", ratio=1),
        Layout(name="footer", size=3),
    )
    layout["header"].update(Panel("Service Health Dashboard", style="bold blue"))
    layout["body"].update(Panel(tbl, title="checks"))
    layout["footer"].update(Panel(f"last refresh: {datetime.now().isoformat()}", style="dim"))
    return layout

console.clear()
console.print(render_panel_layout(checks, refresh_count=0))

## Approach 3 — static snapshots

If the dashboard needs to be archived or compared over time, rendering to a static string is more useful than live terminal output. `console.export_text()` and `console.export_html()` capture the current state without requiring a `Live` display. This approach is heavier per render, but it produces artifacts that can be stored, pasted into chat, or diffed in a PR.

In [ ]:
def render_snapshot(data, refresh_count=0):
    tbl = Table(show_header=True, header_style="bold")
    tbl.add_column("service", style="cyan")
    tbl.add_column("latency_ms", justify="right", style="magenta")
    tbl.add_column("status", justify="center")

    for row in data:
        color = {"ok": "green", "warn": "yellow", "err": "red"}[row["status"]]
        tbl.add_row(
            row["service"],
            str(row["latency_ms"]),
            Text(row["status"], style=color),
        )

    snapshot = Panel(
        tbl,
        title=f"snapshot {refresh_count} — {datetime.now().isoformat()}",
        border_style="blue",
    )
    return snapshot

snap = render_snapshot(checks, refresh_count=0)
console.clear()
console.print(snap)

# Export to plain text for logs or CI artifacts
text = console.export_text(clear=False)
print("--- exported text ---")
print(text)

## Side-by-side comparison

| Approach | Best for | Refresh cost | Export friendliness |
|----------|----------|--------------|---------------------|
| Table | Quick pager-friendly output | Low — single renderable | Medium — export_text works, no layout |
| Panel+Layout | Multi-region dashboard with headers/footers | Medium — full layout redraw | Low — layout nesting does not export cleanly to text |
| Static snapshots | Audit trail, PR comments, Slack posts | Higher — full render + export per frame | High — export_text / export_html give portable artifacts |

The table-only approach is the right default when the output goes to a CI log or a `less` pager. Panel+Layout wins when an operator is watching a terminal and needs context (timestamps, section titles, color-coded borders). Static snapshots are the only approach that produces artifacts without a live terminal.

## Live-refresh loop (optional)

If the data changes, all three approaches can be wrapped in a `Live` loop. The table-only version is the cheapest because it redraws fewer renderables. The panel+layout version redraws the whole split, which is fine on modern terminals but can flicker on slow SSH links. Static snapshots are not meant for live refresh; they are better written to a file on each tick.

In [ ]:
# 5-second live refresh of the table-only approach
with Live(console=console, refresh_per_second=1) as live:
    for i in range(3):
        checks[1]["latency_ms"] = 100 + i * 20
        checks[2]["status"] = "warn" if i % 2 == 0 else "ok"
        live.update(render_table(checks, refresh_count=i))
        time.sleep(1)

console.clear()
console.print("[dim]live loop complete[/dim]")

## Got stuck on

- `Live` with `Layout` needs a fixed `refresh_per_second`. An initial version used `time.sleep(2)` inside the loop and the terminal cursor jumped. Lowering the refresh rate to 1 Hz smoothed it out.
- `console.export_text()` returns the full buffer including previous frames unless the console was created fresh. Reusing a single `Console()` across multiple snapshots appends them together, which is fine for logs but confusing for diffing.
- Panel titles wrap oddly when the terminal is narrower than the content. The docs suggest `expand=False` or adjusting the terminal width, but there is no automatic shrink-to-fit flag.

## What I'd try next

- Add a fourth approach using `rich.progress` bars instead of raw latency numbers to see if visual encoding reduces scan time.
- Write the static snapshots to a Markdown file with embedded HTML (`export_html`) and open it in a browser to compare readability.
- Profile the three approaches with a larger dataset (50+ rows) to measure actual render time differences on a 240-row terminal.